In [2]:
pip install pandas

  Using cached pandas-2.2.3-cp313-cp313-win_amd64.whl.metadata (19 kB)
  Using cached pytz-2025.1-py2.py3-none-any.whl.metadata (22 kB)
  Using cached tzdata-2025.1-py2.py3-none-any.whl.metadata (1.4 kB)
Using cached pandas-2.2.3-cp313-cp313-win_amd64.whl (11.5 MB)
Using cached pytz-2025.1-py2.py3-none-any.whl (507 kB)
Using cached tzdata-2025.1-py2.py3-none-any.whl (346 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import math
import pandas as pd
import numpy as np

# Helper function: Calculate entropy
def calculate_entropy(data):
    elements, counts = np.unique(data, return_counts=True)
    return -sum(
        (count / len(data)) * math.log2(count / len(data))
        for count in counts if count > 0
    )

# Helper function: Calculate information gain
def calculate_information_gain(data, feature, target_attribute):
    total_entropy = calculate_entropy(data[target_attribute])
    values, counts = np.unique(data[feature], return_counts=True)
    weighted_entropy = sum(
        (counts[i] / sum(counts)) * calculate_entropy(
            data[data[feature] == values[i]][target_attribute]
        )
        for i in range(len(values))
    )
    return total_entropy - weighted_entropy

# Recursive function to build the decision tree
def build_tree(data, features, target_attribute):
    labels = data[target_attribute]

    # Base cases
    if len(np.unique(labels)) == 1:  # Pure node
        return labels.iloc[0]
    if not features:  # No features left
        return labels.value_counts().idxmax()

    # Find the best feature to split on
    info_gains = [calculate_information_gain(data, feature, target_attribute) for feature in features]
    best_feature_index = np.argmax(info_gains)
    best_feature = features[best_feature_index]

    # Initialize tree structure
    tree = {best_feature: {}}
    remaining_features = [f for f in features if f != best_feature]

    # Recursively build subtrees for each value of the best feature
    for value in np.unique(data[best_feature]):
        subset = data[data[best_feature] == value]
        subtree = build_tree(subset, remaining_features, target_attribute)
        tree[best_feature][value] = subtree

    return tree

# Function to display the decision tree
def display_tree(tree, depth=0):
    if isinstance(tree, dict):
        for key, value in tree.items():
            print(f"{'|   ' * depth}{key}")
            display_tree(value, depth + 1)
    else:
        print(f"{'|   ' * depth}--> {tree}")

# Main function to demonstrate ID3
def main():
    print("Name: Neel Keshruwala")
    print("En no: 2303031057037")

    # Example dataset
    data = {
        "Outlook": ["Sunny", "Sunny", "Overcast", "Rain", "Rain", "Rain", "Overcast", "Sunny", "Sunny", "Rain", "Sunny", "Overcast", "Overcast", "Rain"],
        "Temperature": ["Hot", "Hot", "Hot", "Mild", "Cool", "Cool", "Cool", "Mild", "Cool", "Mild", "Mild", "Mild", "Hot", "Mild"],
        "Humidity": ["High", "High", "High", "High", "Normal", "Normal", "Normal", "High", "Normal", "Normal", "Normal", "High", "Normal", "High"],
        "Wind": ["FALSE", "TRUE", "FALSE", "FALSE", "FALSE", "TRUE", "TRUE", "FALSE", "FALSE", "FALSE", "TRUE", "TRUE", "FALSE", "TRUE"],
        "PlayTennis": ["No", "No", "Yes", "Yes", "Yes", "No", "Yes", "No", "Yes", "Yes", "Yes", "Yes", "Yes", "No"],
    }
    df = pd.DataFrame(data)

    # Define features and target attribute
    target_attribute = "PlayTennis"
    features = list(df.columns[:-1])

    # Build and print the decision tree
    decision_tree = build_tree(df, features, target_attribute)
    print("\nDecision Tree:")
    display_tree(decision_tree)

# Run the main function
if __name__ == "__main__":
    main()


Name: Neel Keshruwala
En no: 2303031057037

Decision Tree:
Outlook
|   Overcast
|   |   --> Yes
|   Rain
|   |   Wind
|   |   |   FALSE
|   |   |   |   --> Yes
|   |   |   TRUE
|   |   |   |   --> No
|   Sunny
|   |   Humidity
|   |   |   High
|   |   |   |   --> No
|   |   |   Normal
|   |   |   |   --> Yes
